In [2]:
import pandas as pd
import joblib 
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('insurance.csv')
X = df.drop('expenses', axis=1) 
y = df['expenses']


num_features = ['age', 'bmi', 'children']
cat_features = ['sex', 'smoker', 'region']

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])


model = Pipeline([
    ('prep', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

model.fit(X, y)

joblib.dump(model, 'insurance_model.joblib')
print("Model saved successfully as insurance_model.joblib!")


import gradio as gr
import pandas as pd
import joblib

loaded_model = joblib.load('insurance_model.joblib')


def predict(age, sex, bmi, children, smoker, region):
    temp_df = pd.DataFrame([[age, sex, bmi, children, smoker, region]], 
                            columns=['age', 'sex', 'bmi', 'children', 'smoker', 'region'])
    
    prediction = loaded_model.predict(temp_df)[0]
    return f"Estimated Medical Expense: ${prediction:,.2f}"


interface = gr.Interface(
    fn=predict,
    inputs=[
        gr.Number(label="Age", value=25),
        gr.Radio(['male', 'female'], label="Sex"),
        gr.Slider(10, 50, label="BMI", value=24),
        gr.Dropdown([0,1,2,3,4,5], label="Children"),
        gr.Radio(['yes', 'no'], label="Smoker"),
        gr.Dropdown(['southwest', 'southeast', 'northwest', 'northeast'], label="Region")
    ],
    outputs="text",
    title="Insurance AI",
    theme="soft"
)

interface.launch()

Model saved successfully as insurance_model.joblib!
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
